## Import some packages

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import integrate
import os

import ADFWI
from ADFWI.model import AcousticModel
from ADFWI.propagator import AcousticPropagator
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import build_layer_model, wavelet
from ADFWI.view import plot_damp, plot_waveform2D, plot_waveform_trace, plot_waveform_wiggle, plot_wavelet

project_path = "./examples/elastic/Iso-elastic-Layer/data-explosion-acoustic"
for subdir in ("model", "waveform", "survey"):
    os.makedirs(os.path.join(project_path, subdir), exist_ok=True)


## Basic Parameter

In [ ]:
device = "npu:0"         # Specify the CPU/GPU/NPU device
dtype = torch.float32     # Set data type to 32-bit floating point
backend = ADFWI.set_backend(device, dtype=dtype)
ox, oz = 0, 0             # Origin coordinates for x and z directions
nz, nx = 100, 200          # Grid dimensions in z and x directions
dx, dz = 50, 50           # Grid spacing in x and z directions
nt, dt = 1501,0.003       # Time steps and time interval
nabc   = 50               # Thickness of the absorbing boundary layer
f0     = 5               # Initial frequency in Hz
free_surface = True       # Enable free surface boundary condition

## Define the True Velocity Model

In [ ]:
# velocity model 
x    = np.arange(0,nx*dx/1000,dx/1000)
y    = np.arange(0,nz*dz/1000,dz/1000)
step = 1 #km
vel_model = build_layer_model(x, y, step)
vp  = vel_model['vp'].T * 1000
rho = np.ones_like(vp)  * 2000
vs  = np.zeros((nz,nx))

# Initialize the elastic model with parameters and properties
model = AcousticModel(
                    ox,oz,nx,nz,dx,dz,
                    vp,rho,
                    vp_grad = False,rho_grad=False,
                    auto_update_rho=False, auto_update_vp =False, 
                    free_surface=free_surface,
                    abc_type="PML",abc_jerjan_alpha=0.007,
                    nabc=nabc
                    )

# Save the model to a file
model.save(os.path.join(project_path, "model/true_model.npz"))

# Print the model representation
print(model.__repr__())

In [ ]:
# Plot the primary wave velocity (vp) and density (rho) of the model
model._plot_vp_rho(figsize=(12,5),wspace=0.2,cbar_pad_fraction=0.18,cbar_height=0.04,cmap='coolwarm',save_path=os.path.join(project_path,"model/true_vp_vs_rho.png"))

## Define the observed System： Survey = Source + Receiver

In [ ]:
# Define source positions in the model
src_z = np.array([5  for i in range(1,nx-1,5)]) 
src_x = np.array([i  for i in range(1,nx-1,5)])

# Generate wavelet for the source
src_t, src_v = wavelet(nt, dt, f0, amp0=1)  # Create time and wavelet amplitude
src_v = integrate.cumtrapz(src_v, axis=-1, initial=0)  # Integrate wavelet to get velocity
source = Source(nt=nt, dt=dt, f0=f0)  # Initialize source object

for i in range(len(src_x)):
    source.add_source(src_x=src_x[i],src_z=src_z[i],src_wavelet=src_v,src_type="mt",src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))
    # source.add_source(src_x=src_x[i],src_z=src_z[i],src_wavelet=src_v,src_type="mt",src_mt=np.array([[0,0,1],[0,0,0],[0,0,0]]))

In [ ]:
# Define receiver positionsz in the model
rcv_z = np.array([5  for i in range(0,nx,1)])
rcv_x = np.array([j  for j in range(0,nx,1)])

receiver = Receiver(nt=nt, dt=dt)  # Initialize receiver object

# Method 1: Add all receivers at once (commented out)
# receiver.add_receivers(rcv_x=rcv_x, rcv_z=rcv_z, rcv_type='pr')

# Method 2: Loop through each receiver position to add them individually
for i in range(len(rcv_x)):
    receiver.add_receiver(rcv_x=rcv_x[i], rcv_z=rcv_z[i], rcv_type="pr")

In [ ]:
# Create a survey object using the defined source and receiver
survey = Survey(source=source, receiver=receiver)

# Print a representation of the survey object to check its configuration
print(survey.__repr__())

In [ ]:
# Plot the wavelet used in the source
source.plot_wavelet(save_path=os.path.join(project_path, "survey/wavelets.png"))

In [ ]:
# Plot the survey configuration over the velocity model
survey.plot(model.vp, cmap='coolwarm', save_path=os.path.join(project_path, "survey/observed_system.png"))

## Define the propagator & Forward Modeling

In [ ]:
# Initialize the wave propagator using the specified model and survey configuration
F = AcousticPropagator(model,survey)

In [ ]:

damp = F.damp
plot_damp(damp)

In [ ]:
# Perform the forward propagation to record waveforms
record_waveform = F.forward()

# Extract recorded pressure wavefield and particle velocities
rcv_p = record_waveform["p"]  # Recorded pressure wavefield
rcv_u = record_waveform["u"]  # Recorded particle velocity in x-direction
rcv_w = record_waveform["w"]  # Recorded particle velocity in z-direction

# Extract forward wavefields for analysis
forward_wavefield_p = record_waveform["forward_wavefield_p"]  # Forward pressure wavefield
forward_wavefield_u = record_waveform["forward_wavefield_u"]  # Forward particle velocity wavefield in x
forward_wavefield_w = record_waveform["forward_wavefield_w"]  # Forward particle velocity wavefield in z

In [ ]:
# Create a SeismicData object to store observed data from the survey
d_obs = SeismicData(survey)

# Record the waveform data into the SeismicData object
d_obs.record_data(record_waveform)

# Save the recorded data to a specified file
d_obs.save(os.path.join(project_path, "waveform/obs_data.npz"))

## Visulization the Synthetic Waveform

In [ ]:
normalize = True  # Set normalization for waveform plotting

# Loop over shots to plot the observed waveforms
for i_shot in range(1,2):  # Currently set to plot only the first shot
    show = (i_shot == 1)  # Show the plot only for the first shot

    # Plot 2D waveform for the specified shot
    d_obs.plot_waveform2D(
        i_shot=i_shot,
        rcv_type="pressure",
        acoustic_or_elastic="acoustic",
        normalize=normalize,
        figsize=(6, 6),
        cmap='coolwarm',
        save_path=os.path.join(project_path, f"waveform/obs_2D_shot_{i_shot}.png"),
        show=show
    )

    # Plot wiggle representation of the waveform for the specified shot
    d_obs.plot_waveform_wiggle(
        i_shot=i_shot,
        rcv_type="pressure",
        acoustic_or_elastic="acoustic",
        normalize=normalize,
        save_path=os.path.join(project_path, f"waveform/obs_wiggle_shot_{i_shot}.png"),
        show=show
    )

In [ ]:
# Plot the waveform trace for a specific shot and trace index
d_obs.plot_waveform_trace(
    i_shot=0,  # Index of the shot to plot (first shot)
    i_trace=10,  # Index of the trace to plot (10th trace)
    normalize=True  # Normalize the waveform for better visualization
)

In [ ]:
from scipy.signal import stft
src_t, src_v = wavelet(nt, dt, f0, amp0=1)  # Create time and wavelet amplitude
fs = int(1/dt)
# 1. 快速傅里叶变换 (FFT)
fft_result = np.fft.fft(src_v)  # FFT 结果（复数）
fft_amplitude = np.abs(fft_result)  # 计算幅值
frequencies = np.fft.fftfreq(len(src_v), 1 / fs)  # 对应的频率

# 2. 只保留正频率部分
positive_freqs = frequencies[frequencies >= 0]
positive_amplitude = fft_amplitude[frequencies >= 0]

wn1 = 1500
sampling_time = dt
pa = np.mean(np.abs(np.fft.rfft(rcv_p[0,:,:].cpu().detach().numpy(),axis=0)),axis=1)[:wn1//2]
fx = np.fft.fftfreq(wn1,sampling_time)[:wn1//2]

# 3. 绘制频率分布图
plt.figure(figsize=(10, 6))
plt.plot(positive_freqs, positive_amplitude/positive_amplitude.max(), label='Frequency Distribution')
plt.plot(fx, pa/pa.max(), label='Frequency Distribution')
plt.xlim(0,20)
plt.title("Frequency Distribution of Seismic Data")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.grid()
plt.legend()
plt.show()